In [0]:
df_bronze = spark.table("bigdata.default.bronze_sales_raw")
print("✅ Registros en Bronze:", df_bronze.count())
df_bronze.show(5)

In [0]:
from pyspark.sql import functions as F

df_silver = df_silver \
    .filter(F.col("CustomerID").isNotNull()) \
    .filter(~F.col("InvoiceNo").startswith("C")) \
    .filter(F.col("Quantity") > 0) \
    .filter(F.col("UnitPrice") > 0)

#  Eliminar registros sin CustomerID
#como no sabemos quien compró no podemos segmentar
# Filtrar cantidades negativas (devoluciones)
#quitamos facturas canceladas 

# Renombrar columnas al español
df_silver = df_silver \
    .withColumnRenamed("InvoiceNo",   "numero_factura") \
    .withColumnRenamed("StockCode",   "codigo_producto") \
    .withColumnRenamed("Description", "descripcion") \
    .withColumnRenamed("Quantity",    "cantidad") \
    .withColumnRenamed("InvoiceDate", "fecha_factura") \
    .withColumnRenamed("UnitPrice",   "precio_unitario") \
    .withColumnRenamed("CustomerID",  "id_cliente") \
    .withColumnRenamed("Country",     "pais")
# Calcular TotalAmount
df_silver = df_silver.withColumn(
    "TotalAmount",
    F.col("cantidad") * F.col("precio_unitario")
)

print("🥈 Silver registros:", df_silver.count())
df_silver.show(5)

# Guardar tabla
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bigdata.default.silver_sales_cleaned")

print("✅ silver_sales_cleaned guardada!")

In [0]:
bronze = spark.table("bigdata.default.bronze_sales_raw").count()
silver = spark.table("bigdata.default.silver_sales_cleaned").count()

print(f"🥉 Bronze: {bronze:,} registros")
print(f"🥈 Silver: {silver:,} registros")
print(f"🗑️  Eliminados: {bronze - silver:,} registros sucios")
